In [1]:
import pdfplumber

In [ ]:
import pdfplumber

def extract_text(file_path):
    text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text.lower()

def extract_skills(text):
    skills_db = [

    # 🔹 Programming Languages
    "python", "java", "c", "c++", "javascript", "typescript", "go", "ruby", "php",

    # 🔹 Web Development
    "html", "css", "react", "angular", "vue", "node", "express", "next.js",
    "bootstrap", "tailwind", "rest api", "graphql",

    # 🔹 Data Science / AI / ML
    "machine learning", "deep learning", "nlp", "computer vision",
    "data analysis", "data science", "pandas", "numpy", "matplotlib",
    "seaborn", "scikit-learn", "tensorflow", "pytorch", "keras",

    # 🔹 Databases
    "sql", "mysql", "postgresql", "mongodb", "firebase", "redis",

    # 🔹 Cloud / DevOps
    "aws", "azure", "gcp", "docker", "kubernetes", "ci/cd", "jenkins",
    "github actions", "linux", "bash", "terraform",

    # 🔹 Mobile Development
    "android", "kotlin", "flutter", "react native", "swift",

    # 🔹 Tools & Others
    "git", "github", "bitbucket", "jira", "postman", "figma",

    # 🔹 Cybersecurity
    "network security", "ethical hacking", "penetration testing",

    # 🔹 Core CS
    "data structures", "algorithms", "operating system", "dbms", "computer networks"
]
    found = [skill for skill in skills_db if skill in text]
    return found


resume_path = "new.pdf"   # put your resume file here

text = extract_text(resume_path)
skills = extract_skills(text)




In [ ]:
def detect_domain(skills):

    if any(s in skills for s in ["machine learning", "deep learning", "nlp", "tensorflow", "pytorch"]):
        return "AI/ML"

    elif any(s in skills for s in ["react", "angular", "vue", "node", "html", "css"]):
        return "Web Development"

    elif any(s in skills for s in ["sql", "mongodb", "data analysis", "pandas"]):
        return "Data Science"

    elif any(s in skills for s in ["android", "flutter", "react native"]):
        return "Mobile Development"

    elif any(s in skills for s in ["aws", "docker", "kubernetes"]):
        return "Cloud/DevOps"

    elif any(s in skills for s in ["network security", "ethical hacking"]):
        return "Cybersecurity"

    else:
        return "General Software"
    

In [19]:
resume_path = "new.pdf"

text = extract_text(resume_path)
skills = extract_skills(text)

domain = detect_domain(skills)
print("Extracted Skills:", skills)
print("Detected Domain:", domain)

Extracted Skills: ['python', 'c', 'c++', 'go', 'machine learning', 'deep learning', 'nlp', 'data analysis', 'pandas', 'numpy', 'matplotlib', 'scikit-learn', 'tensorflow', 'pytorch', 'keras', 'sql', 'git', 'github']
Detected Domain: AI/ML


In [ ]:
from langchain_groq import ChatGroq


In [ ]:
LLM = ChatGroq(
    model="llama3-8b-8192",
    api_key="GROQ_API_KEY"
)

In [9]:
def generate_question(domain,skills):
  def skill_node(state):
    text = state["text"]

    res = llm.invoke(f"Extract only technical skills from this resume:\n{text}")
    return {"skills": res.content}


# ---- Domain Detection ----
def domain_node(state):
    skills = state["skills"]

    res = LLM.invoke(f"Identify domain from these skills: {skills}")
    return {"domain": res.content}


# ---- Question Generation ----
def question_node(state):
    skills = state["skills"]
    domain = state["domain"]

    res = LLM.invoke(
        f"Generate 5 interview questions for skills {skills} in domain {domain}"
    )
    return {"questions": res.content}


# ---- Answer Evaluation ----
def evaluation_node(state):
    questions = state["questions"]
    answer = state.get("answer", "Sample answer")

    res = LLM.invoke(
        f"Evaluate this answer:\nQ:{questions}\nA:{answer}\nGive score out of 10 + feedback"
    )
    return {"evaluation": res.content}
    